# LangChain Expression Language (LCEL)
### Compose chains with `|`, run them in parallel, and branch on conditions

---
**Topics Covered:**
- The `|` pipe operator — connecting runnables
- `RunnableParallel` — run independent chains at the same time
- `RunnableLambda` — wrap any Python function as a runnable
- `RunnablePassthrough` — forward inputs unchanged
- `RunnableBranch` — conditional routing
- Inspecting chain schemas with `.input_schema` and `.output_schema`


In [15]:
import os
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings('ignore')

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough, RunnableBranch

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq", temperature=0.5)

## 1. The Pipe Operator `|`

Every LangChain component that implements `Runnable` can be chained with `|`.  
Data flows left → right through each step.

In [16]:
# A simple 3-step chain
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical summariser. Be concise."),
    ("human", "Summarise the following topic in exactly 3 bullet points:\n\n{topic}")
])

summarize_chain = summarize_prompt | llm | StrOutputParser()

output = summarize_chain.invoke({"topic": "Transformer architecture in deep learning"})
print(output)

* The Transformer architecture is a type of neural network introduced in 2017, primarily designed for sequence-to-sequence tasks such as machine translation, relying on self-attention mechanisms to weigh input elements.
* The Transformer model consists of an encoder and a decoder, with the encoder taking in a sequence of tokens and the decoder generating output tokens, using multi-head attention and feed-forward neural networks to process input sequences.
* The Transformer architecture has become a cornerstone in natural language processing, achieving state-of-the-art results in various tasks, and has also been applied to other areas such as computer vision and speech recognition, due to its ability to handle long-range dependencies and parallelize computation.


## 2. `RunnableLambda` — Python Functions as Runnables

In [17]:
# Post-processing step: count words in the output
word_counter = RunnableLambda(lambda text: {
    "summary": text,
    "word_count": len(text.split())
})

# Extend the chain with a custom Python step
extended_chain = summarize_chain | word_counter

result = extended_chain.invoke({"topic": "Gradient descent optimisation"})
print("Word count:", result["word_count"])
print("Summary:\n",  result["summary"])

Word count: 81
Summary:
 * Gradient descent is an optimisation algorithm used to minimize the loss function in various machine learning models by iteratively adjusting the model's parameters.
* The algorithm works by calculating the gradient of the loss function with respect to the model's parameters, then updating the parameters in the direction that reduces the loss.
* Gradient descent has several variants, including batch gradient descent, stochastic gradient descent, and mini-batch gradient descent, each with its own trade-offs between computational efficiency and convergence speed.


## 3. `RunnablePassthrough` — Pass Input Alongside

Use `RunnablePassthrough.assign(key=chain)` to add new fields to a dict while keeping original fields intact.

In [18]:
# Keep the original 'topic' AND add the LLM's summary
augment_chain = (
    RunnablePassthrough.assign(
        summary=summarize_chain
    )
)

out = augment_chain.invoke({"topic": "RLHF — reinforcement learning from human feedback"})
print("Original input:", out["topic"])
print("\nAdded summary:\n", out["summary"])

Original input: RLHF — reinforcement learning from human feedback

Added summary:
 * RLHF (Reinforcement Learning from Human Feedback) is a machine learning approach that involves training models using human feedback and reinforcement signals.
* The process involves a human evaluator providing feedback on the model's output, which is then used to update the model's parameters and improve its performance over time.
* RLHF has been used to fine-tune large language models, enabling them to generate more accurate and relevant responses, and has shown promising results in areas such as natural language processing and conversational AI.


## 4. `RunnableParallel` — Fan-Out to Multiple Chains

Run independent chains concurrently and collect their outputs in a single dict.

In [19]:
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List only PROS. Use 3 bullet points max."),
    ("human", "Topic: {topic}")
])

cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List only CONS. Use 3 bullet points max."),
    ("human", "Topic: {topic}")
])

pros_chain = pros_prompt | llm | StrOutputParser()
cons_chain = cons_prompt | llm | StrOutputParser()

# Both chains receive the same input and run concurrently
pros_cons = RunnableParallel(pros=pros_chain, cons=cons_chain)

result = pros_cons.invoke({"topic": "Microservices architecture"})
print("PROS:\n", result["pros"])
print("\nCONS:\n", result["cons"])

PROS:
 Here are the pros of microservices architecture:
* **Improved scalability**: Microservices allow for individual components to be scaled independently, reducing the need to scale the entire application.
* **Enhanced fault tolerance**: If one microservice experiences issues, it won't bring down the entire application, as other services can continue to function independently.
* **Increased flexibility**: Microservices enable the use of different programming languages, frameworks, and databases for each service, allowing for greater flexibility and innovation.

CONS:
 * Increased complexity: Microservices architecture can lead to increased complexity, making it harder to manage and maintain the system as a whole.
* Higher operational overhead: With multiple services to manage, there is a higher operational overhead, including more servers to maintain, more code to update, and more potential points of failure.
* Greater communication challenges: Since microservices are designed to be

## 5. Chaining Parallel Results into a Final Step

In [20]:
combine_prompt = ChatPromptTemplate.from_messages([
    ("system", "You synthesise pros/cons into an actionable recommendation."),
    ("human", "PROS:\n{pros}\n\nCONS:\n{cons}\n\nGive a 2-sentence recommendation.")
])

full_pipeline = pros_cons | combine_prompt | llm | StrOutputParser()

recommendation = full_pipeline.invoke({"topic": "Microservices architecture"})
print(recommendation)

I recommend adopting a microservices architecture for applications that require high scalability, flexibility, and resilience, as the benefits of independent scaling and fault tolerance outweigh the increased complexity and operational overhead. However, it's essential to carefully weigh the pros and cons and consider implementing robust monitoring, logging, and debugging tools to mitigate the potential risks and challenges associated with microservices architecture.


## 6. `RunnableBranch` — Conditional Routing

Route to different chains based on a condition evaluated on the input.

In [21]:
# Tone-aware responder: formal vs casual based on 'tone' key
formal_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional assistant. Respond formally."),
    ("human", "{question}")
])

casual_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly chatbot. Respond casually with emojis."),
    ("human", "{question}")
])

formal_chain = formal_prompt | llm | StrOutputParser()
casual_chain = casual_prompt | llm | StrOutputParser()

tone_branch = RunnableBranch(
    (lambda x: x.get("tone") == "formal",  formal_chain),
    casual_chain    # default branch
)

print("--- Formal ---")
print(tone_branch.invoke({"question": "What is recursion?", "tone": "formal"}))

print("\n--- Casual ---")
print(tone_branch.invoke({"question": "What is recursion?", "tone": "casual"}))

--- Formal ---
Recursion is a fundamental concept in computer science and mathematics that refers to a method or function that invokes itself repeatedly until it reaches a base case that stops the recursion. In other words, a recursive function solves a problem by breaking it down into smaller instances of the same problem, which are then solved by the same function, until the solution to the original problem is found.

The key elements of recursion are:

1. **Base case**: A trivial case that can be solved directly, without calling the function again. This case serves as a stopping point for the recursion.
2. **Recursive case**: A case that requires the function to call itself to solve the problem. The function breaks down the problem into smaller instances, and then calls itself to solve these instances.
3. **Recursive call**: The function calls itself, either directly or indirectly, to solve a smaller instance of the problem.

Recursion can be used to solve a wide range of problems, 

## 7. Inspect Chain Schemas

In [22]:
import json
print("Input schema:")
print(json.dumps(summarize_chain.input_schema.model_json_schema(), indent=2))

print("\nOutput schema:")
print(json.dumps(summarize_chain.output_schema.model_json_schema(), indent=2))

Input schema:
{
  "properties": {
    "topic": {
      "title": "Topic",
      "type": "string"
    }
  },
  "required": [
    "topic"
  ],
  "title": "PromptInput",
  "type": "object"
}

Output schema:
{
  "title": "StrOutputParserOutput",
  "type": "string"
}


## 8. Chain Introspection with `.get_graph()`

In [23]:
pip install grandalf

Note: you may need to restart the kernel to use updated packages.


In [24]:
graph = full_pipeline.get_graph()
graph.print_ascii()

ImportError: Install grandalf to draw graphs: `pip install grandalf`.

---
### Summary

| Runnable | What it does |
|----------|--------------|
| `A \| B \| C` | Sequential pipeline |
| `RunnableLambda(fn)` | Any Python function as a step |
| `RunnablePassthrough.assign(key=chain)` | Augment input dict with new field |
| `RunnableParallel(a=chain1, b=chain2)` | Run concurrently, merge outputs |
| `RunnableBranch((cond, chain), default)` | Conditional routing |

**Next → `05_streaming_and_batch.ipynb`**